In [ ]:
using LinearAlgebra
using Plots

## Questão 2 )



### Letra a)

Pelo exercício 1, dado $m \geq  d + 1$, podemos garantir que $L_X$ possui posto completo, o que implica que $L_X$ é uma transformação injetiva. Ou seja, garantimos que suas inversas são inversas à esquerda. 

Sabemos que "$R_{X,d} \circ L_X = id_\mathcal{P_d}$ somente se $R_{X,d} \circ L_X(p) = p$ para todo $p \in \mathcal{P_d}$"

Pela definição, sabemos que $R_{X,d}(L_X(p)) = \argmin_{q \in \mathcal{P}_d}\|L_X(q) - L_X(p)\|^2_2$. Isso é minimizado quando $$\|L_X(q) - L_X(p)\|^2_2 = 0$$, o que acontece quando $L_X(q) = L_X(p)$. Porém, como $L_X$ é injetiva, $p = q$. Assim, para todo $p \in \mathcal{P_d}$, $R_{X,d} \circ L_X(p) = p$ provando que $R_{X,d}$ é inversa à esquerda de $L_X$.



### Letra b)

Vamos adotar a matriz de Vandermonde associada a $L_X$. Pela definição, temos que resolver o problema de mínimos quadrados, o que envolve encontrar

$$ A^*Ac = A^*b \rightarrow c = (A^*A)^{-1}A^*b$$

Podemos ver que a transformação depende linearmente de $b$. Logo, $R_{X,d}$ é uma transfomação linear.

### Letra c)

Sabemos que na base canônica de $\mathcal{P_d}$, $\lbrace 1, x, x^2 \dots x^d \rbrace$, nossa matriz $L_X$ se torna a matriz de Vandermonde retangular, já que sabemos que $m \geq d + 1$.


$$ \textit{Matriz de Vandermonde (V)} :
\begin{bmatrix}
1 & x_1 & x_1^2 & \dots & x_1^d \\
1 & x_2 & x_2^2 & \dots & x_2^d \\
\vdots & \vdots & \vdots & \vdots & \vdots \\
1 & x_m & x_m^2 & \dots & x_m^d 
\end{bmatrix}
$$

Como vimos no item $b$, a solução de mínimos quadrados é $c = (A^*A)^{-1}A^*b$. Portanto, podemos calcular $R_{X,d}$ como a pseudo-inversa de $L_x$. Assim,

$$ R_{X,d} = (V^*V)^{-1}V^*$$

### Letra d )

A questão pede para escolhermos alguns valores de $m =$ #$X$ e $d$ e estudar qual o comportamento do condicionamento de $L_X$ e $R_{X,d}$ na base canônica $\mathcal{P_d}$, ou seja, $L_X$ é uma matriz de Vandermonde retangular e $R_{X,d}$ sua pseudo-inversa, quando $m$ e/ou $d$ aumentam.

In [ ]:
function createVandermonde(vetor, d)
    Vandermonde = vetor .^ (0:d)'
    return Vandermonde
end    

function evaluate_condition(m, d)
    vetor = LinRange(-1,1,m)
    V = createVandermonde(vetor, d)
    return cond(V)
end

In [ ]:
function plot_condition_evaluated(m, d, variate_m = true, variate_d = false, log_scale= false)
    if variate_m && !variate_d
        x = d+1:m
        y = [evaluate_condition(i, d) for i in x]

        if log_scale
            display(plot(x, y, xlabel= "m", ylabel= "condition", title= "d fixed = $d, m ascending", yscale= :log10))
        else
            display(plot(x, y, xlabel= "m", ylabel= "condition", title= "d fixed = $d, m ascending"))
        end
        
    elseif !variate_m && variate_d
        x = 1:d
        y = [evaluate_condition(m, j) for j in x]

        if log_scale
            display(plot(x, y, label="m fixo=$m", xlabel= "d", ylabel= "condition", title= "m fixed = $m, d ascending", yscale= :log10))
        else
            display(plot(x, y, label="m fixo=$m", xlabel= "d", ylabel= "condition", title= "m fixed = $m, d ascending"))
        end

    elseif variate_d && variate_m
        # plt = plot()
        # for j in 1:d
        #     x = 2:m
        #     y = [evaluate_condition(i, j) for i in x]

        #     if log_scale
        #         plot!(plt, x, y, label="d=$j", leg= :topright, yscale= :log10)
        #     else
        #         plot!(plt, x, y, label="d=$j", leg= :topright)
        #     end
        # end
        
        data = Float64[]
        data_j = Int8[]
        j = 1
        for i in j+1:m
            j = j + div(i,(m/5))
            push!(data_j, j)
            push!(data, evaluate_condition(i, j))
        end

        print(unique(data_j))
        plot(2:m, data)
    # ylabel!("condition")
    # xlabel!("m")
    # display(plt)
    end
end

In [ ]:
function plot_condition_evaluated(m, d, variate_m=true, variate_d=false, log_scale=false)
    if variate_m && !variate_d
        x = (d+1):m
        y = [evaluate_condition(i, d) for i in x]
        ylab = log_scale ? "condition (log)" : "condition"
        display(plot(x, y, xlabel="m", ylabel=ylab,
                     title="d fixo = $d, m crescendo",
                     yscale=log_scale ? :log10 : :identity))

    elseif !variate_m && variate_d
        x = 1:(m-1)  # d < m para Vandermonde ter posto completo
        y = [evaluate_condition(m, j) for j in x]
        ylab = log_scale ? "condition (log)" : "condition"
        display(plot(x, y, xlabel="d", ylabel=ylab,
                     title="m fixo = $m, d crescendo",
                     yscale=log_scale ? :log10 : :identity))

    elseif variate_m && variate_d
        # Heatmap: eixo x = m, eixo y = d
        ms = 3:m
        ds = 1:(m-2)
        # só calcula onde m >= d+1
        Z = [mi >= di+1 ? evaluate_condition(mi, di) : NaN
             for di in ds, mi in ms]

        if log_scale
            Z = log10.(Z)
        end

        display(heatmap(ms, ds, Z,
                        xlabel="m", ylabel="d",
                        title=log_scale ? "log10(cond(V))" : "cond(V)",
                        color=:viridis))
    end
end

Apenas $m$ aumentando

In [ ]:
plot_condition_evaluated(1000, 2, true, false)


In [ ]:
plot_condition_evaluated(1000, 10, true, false)

In [ ]:
# plot_condition_evaluated(1000000, 10, false, true)


plot_condition_evaluated(1000, 10, false, true)

In [ ]:
# plot_condition_evaluated(100000, 5, true, true)

### Letra e)

Sabemos que a matriz de Vandermonde realiza a avalização de um polinômio da base canônica em um ponto $\alpha_i$. Trocando da base cancônica para a base dos polinômios interpoladores de Lagrange, sabemos que 

$$p_i(x_j) = \delta_{ij},  \space \space 
 \delta{ij} =\begin{cases}
                1, i=j \\
                0, i \neq j
                \end{cases}
$$

Assim, ao avaliar um polinômio que é a combinação linear dos polinômios da base em um ponto $x_j$ (linha $j$), aparecerá apenas a parte do polinômio relacionada àquela linha. Logo, $L_X$ na base de Lagrange é

$$
\begin{bmatrix}
I_{d+1 \times d+1} \\ 0_{m - (d+1) \times d+1}
\end{bmatrix}

\Longrightarrow 

(L_X)_{ij} = \begin{cases}
                1, i=j \\
                0, i \neq j
                \end{cases}
$$

* Essa base foi calculada com $p_i(x) = \prod_{\substack{j=0 \\ j \neq i}}^{n} \frac{x - x_j}{x_i - x_j}$, fazendo com que não tivesse lixo, diferente da interpolação utilizada para provar que $L_X$ tem posto completo.

Assim, $R_X,d$ nessa nova base tem a mesma fórmula que na base antiga.


In [ ]:
diagm(10, 3, [1,1,1])

In [ ]:
using LinearAlgebra
using Plots

function matriz_LX(X, d)
    m = length(X)
    # Matriz de Vandermonde m x (d+1)
    V = [x^j for x in X, j in 0:d]
    return V
end

function matriz_RXd(X, d)
    V = matriz_LX(X, d)
    return pinv(V)  # pseudoinversa (d+1) x m
end

# Variando d com m fixo
m = 20
graus = 1:15
conds_V  = [cond(matriz_LX(range(-1, 1, m), d)) for d in graus]
conds_R  = [cond(matriz_RXd(range(-1, 1, m), d)) for d in graus]

p1 = plot(graus, conds_V, yscale=:log10, label="L_X",
          marker=:circle, xlabel="grau d",
          ylabel="condicionamento (log)", title="m=$m fixo")
plot!(p1, graus, conds_R, yscale=:log10, label="R_{X,d}",
      marker=:square, linestyle=:dash)

# Variando m com d fixo
d = 5
ms = (d+1):30
conds_V2 = [cond(matriz_LX(range(-1, 1, m), d)) for m in ms]
conds_R2 = [cond(matriz_RXd(range(-1, 1, m), d)) for m in ms]

p2 = plot(ms, conds_V2, yscale=:log10, label="L_X",
          marker=:circle, xlabel="m (pontos)",
          ylabel="condicionamento (log)", title="d=$d fixo")
plot!(p2, ms, conds_R2, yscale=:log10, label="R_{X,d}",
      marker=:square, linestyle=:dash)

plot(p1, p2, layout=(1,2), size=(900, 400))